<a href="https://colab.research.google.com/github/Stf12-creator/01/blob/main/ComfyUI%20Colab%20Upgraded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ ComfyUI: Professional Edition

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

### 🖥️ **High-Performance AI Environment**
This notebook provides a stable, persistent, and optimized environment for running ComfyUI on Google Colab.

**Key Features:**
- **Tarball SSD Caching:** UI nodes are packed into a single high-speed archive to bypass Google Drive's small-file bottleneck.
- **Modern Downloader:** Python-native, high-speed downloader with visual progress bars.
- **Memory Safety:** Prevents OOM (Out of Memory) errors via selectable GPU profiles.

In [1]:
#@title 1. System Initialization
#@markdown **Run this cell first.** <br>
#@markdown Mounts Drive, sets up ComfyUI, and extracts your high-speed Tarball cache to the local SSD for instant booting.

import os
import shutil
import subprocess
import sys
from google.colab import drive

print("[SYSTEM] Initialization Protocol Started...\n")

# --- Configuration ---
MOUNT_DRIVE = True
UPDATE_COMFY_UI = True #@param {type:"boolean"}
INSTALL_COMFYUI_MANAGER = True #@param {type:"boolean"}

LOCAL_WORKSPACE = "/content/ComfyUI"
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI"
CACHE_TAR = os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar")

def stream_cmd(cmd, cwd=None):
    """Streams shell commands directly to the output"""
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd, bufsize=1)
    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()

# 1. Mount Google Drive
if MOUNT_DRIVE:
    print("💾 Requesting Google Drive Access (Check popup if not already mounted)...")
    drive.mount('/content/drive')

# 2. Setup ComfyUI Core
if not os.path.exists(LOCAL_WORKSPACE):
    print("\n📦 Cloning ComfyUI repository...")
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", LOCAL_WORKSPACE])
else:
    if UPDATE_COMFY_UI:
        print("\n🔄 Checking for ComfyUI updates...")
        subprocess.run(["git", "pull"], cwd=LOCAL_WORKSPACE)

# 3. Configure Hybrid Storage & Tarball Cache
heavy_dirs = ["models", "output", "input"]
print("\n🔗 Routing heavy model directories directly to Drive...")
for d in heavy_dirs:
    local_path = os.path.join(LOCAL_WORKSPACE, d)
    drive_path = os.path.join(DRIVE_WORKSPACE, d)
    if not os.path.exists(drive_path): os.makedirs(drive_path, exist_ok=True)
    if os.path.exists(local_path) and not os.path.islink(local_path): shutil.rmtree(local_path)
    if not os.path.exists(local_path): os.symlink(drive_path, local_path)

print("\n⚡ Deploying High-Speed Tarball Cache for UI Elements...")
if os.path.exists(CACHE_TAR):
    print("   📦 Extracting cached UI nodes to local SSD (Superfast)...")
    os.system(f"tar -xf '{CACHE_TAR}' -C '{LOCAL_WORKSPACE}' > /dev/null 2>&1")
else:
    print("   ⚠️ No cache found on Drive. Creating fresh local directories...")
    os.makedirs(os.path.join(LOCAL_WORKSPACE, "custom_nodes"), exist_ok=True)
    os.makedirs(os.path.join(LOCAL_WORKSPACE, "user"), exist_ok=True)

# 4. Install ComfyUI Manager
if INSTALL_COMFYUI_MANAGER:
    manager_path = os.path.join(LOCAL_WORKSPACE, "custom_nodes", "ComfyUI-Manager")
    if not os.path.exists(manager_path):
        print("\n📦 Installing ComfyUI Manager...")
        subprocess.run(["git", "clone", "https://github.com/ltdrdata/ComfyUI-Manager.git", manager_path])
    else:
        print("\n✅ ComfyUI Manager verified.")
        subprocess.run(["git", "pull"], cwd=manager_path)

# 5. Core Python Dependencies
print("\n🛠️ Installing base Python requirements...")
stream_cmd(["pip", "install", "xformers!=0.0.18", "-r", "requirements.txt", "--extra-index-url", "https://download.pytorch.org/whl/cu121"], cwd=LOCAL_WORKSPACE)
stream_cmd(["pip", "install", "insightface", "onnxruntime-gpu"])

# 6. AUTO-HEAL: Custom Node Dependencies
print("\n🔍 [AUTO-HEAL] Scanning local custom nodes for missing dependencies...")
custom_nodes_dir = os.path.join(LOCAL_WORKSPACE, "custom_nodes")
if os.path.exists(custom_nodes_dir):
    for item in os.listdir(custom_nodes_dir):
        node_path = os.path.join(custom_nodes_dir, item)
        req_file = os.path.join(node_path, "requirements.txt")
        if os.path.isdir(node_path) and os.path.exists(req_file):
            print(f"\n   ⚙️ Restoring dependencies for: {item}")
            stream_cmd(["pip", "install", "-r", "requirements.txt"], cwd=node_path)

print("\n✅ [SYSTEM READY] Initialization complete.")

[SYSTEM] Initialization Protocol Started...

💾 Requesting Google Drive Access (Check popup if not already mounted)...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🔄 Checking for ComfyUI updates...

🔗 Routing heavy model directories directly to Drive...

⚡ Deploying High-Speed Tarball Cache for UI Elements...
   ⚠️ No cache found on Drive. Creating fresh local directories...

✅ ComfyUI Manager verified.

🛠️ Installing base Python requirements...
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121

🔍 [AUTO-HEAL] Scanning local custom nodes for missing dependencies...

   ⚙️ Restoring dependencies for: comfyui-promptchain

   ⚙️ Restoring dependencies for: ComfyUI-Manager

✅ [SYSTEM READY] Initialization complete.


In [2]:
#@title 2. Model & Node Downloader
#@markdown Paste download links below. This supports **Checkpoints, Diffusion Models (Flux/UNET), Text Encoders (CLIP/T5), CLIP Vision, VAEs, LoRAs, and ControlNets**.

import os
import requests
from urllib.parse import urlparse, unquote
from tqdm.auto import tqdm

WORKSPACE = "/content/ComfyUI"

# --- Input Resources ---
CHECKPOINT_URLS = "https://huggingface.co/SG161222/RealVisXL_V4.0/resolve/main/RealVisXL_V4.0.safetensors" #@param {type:"string"}
UNET_DIFFUSION_URLS = "" #@param {type:"string"}
TEXT_ENCODER_URLS = "" #@param {type:"string"}
CLIP_VISION_URLS = "https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors" #@param {type:"string"}
VAE_URLS = "" #@param {type:"string"}
LORA_URLS = "" #@param {type:"string"}
CONTROLNET_URLS = "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/diffusers_xl_canny_mid.safetensors" #@param {type:"string"}
UPSCALE_MODELS_URLS = "" #@param {type:"string"}
EMBEDDING_URLS = "" #@param {type:"string"}
CUSTOM_NODE_URLS = "" #@param {type:"string"}

# --- Downloader Logic ---
DIRS = {
    "checkpoints":    os.path.join(WORKSPACE, "models/checkpoints"),
    "unet":           os.path.join(WORKSPACE, "models/unet"),
    "clip":           os.path.join(WORKSPACE, "models/clip"),
    "clip_vision":    os.path.join(WORKSPACE, "models/clip_vision"),
    "vae":            os.path.join(WORKSPACE, "models/vae"),
    "loras":          os.path.join(WORKSPACE, "models/loras"),
    "controlnet":     os.path.join(WORKSPACE, "models/controlnet"),
    "upscale_models": os.path.join(WORKSPACE, "models/upscale_models"),
    "embeddings":     os.path.join(WORKSPACE, "models/embeddings"),
    "custom_nodes":   os.path.join(WORKSPACE, "custom_nodes")
}

def get_filename(url, response):
    """Smartly determines filename from Content-Disposition or URL."""
    if "Content-Disposition" in response.headers:
        import re
        fname = re.findall('filename="?([^"]+)"?', response.headers["Content-Disposition"])
        if fname: return fname[0]
    return unquote(os.path.basename(urlparse(url).path))

def download_file(url, target_dir):
    try:
        # Stream the download to get headers first
        response = requests.get(url, stream=True, allow_redirects=True)
        response.raise_for_status()

        filename = get_filename(url, response)
        file_path = os.path.join(target_dir, filename)
        total_size = int(response.headers.get('content-length', 0))

        if os.path.exists(file_path):
            print(f"   ⏩ Skipping (Exists): {filename}")
            return

        # Modern Progress Bar Log
        print(f"   📥 Downloading: {filename}")

        # The Progress Bar (Auto-Stretching)
        with tqdm(
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
            desc="      🚀 Progress",
            dynamic_ncols=True
        ) as bar:
            with open(file_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024*1024): # 1MB chunks
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

        print("      ✅ Download Complete\n")

    except Exception as e:
        print(f"   ❌ Failed to download: {url}")
        print(f"      Error: {e}\n")

def process_downloads(urls_str, target_dir, is_node=False):
    if not urls_str.strip(): return

    url_list = [u.strip() for u in urls_str.replace(',', '\n').split('\n') if u.strip()]
    if not os.path.exists(target_dir): os.makedirs(target_dir, exist_ok=True)

    print(f"📂 Category: {os.path.basename(target_dir)}")

    for url in url_list:
        if is_node:
            node_name = url.split('/')[-1].replace('.git', '')
            node_path = os.path.join(target_dir, node_name)
            if not os.path.exists(node_path):
                print(f"   ⬇️ Cloning Node: {node_name}...")
                !git clone {url} {node_path}
                # Auto-install requirements
                req = os.path.join(node_path, "requirements.txt")
                if os.path.exists(req):
                    print(f"      📦 Installing requirements...")
                    !pip install -r "{req}"
                print("      ✅ Installed\n")
            else:
                print(f"   ⏩ Node exists: {node_name}\n")
        else:
            download_file(url, target_dir)

# --- Execution ---
process_downloads(CHECKPOINT_URLS,     DIRS["checkpoints"])
process_downloads(UNET_DIFFUSION_URLS, DIRS["unet"])
process_downloads(TEXT_ENCODER_URLS,   DIRS["clip"])
process_downloads(CLIP_VISION_URLS,    DIRS["clip_vision"])
process_downloads(VAE_URLS,            DIRS["vae"])
process_downloads(LORA_URLS,           DIRS["loras"])
process_downloads(CONTROLNET_URLS,     DIRS["controlnet"])
process_downloads(UPSCALE_MODELS_URLS, DIRS["upscale_models"])
process_downloads(EMBEDDING_URLS,      DIRS["embeddings"])
process_downloads(CUSTOM_NODE_URLS,    DIRS["custom_nodes"], is_node=True)

print("🎉 All tasks finished.")

📂 Category: checkpoints
   ⏩ Skipping (Exists): RealVisXL_V4.0.safetensors
📂 Category: clip_vision
   ⏩ Skipping (Exists): model.safetensors
📂 Category: controlnet
   ⏩ Skipping (Exists): diffusers_xl_canny_mid.safetensors
🎉 All tasks finished.


In [3]:
# @title 3. Session Anti-Disconnect
%%html
<b>🔊 Keep-Alive Audio</b><br>
<i>Running this silent audio loop prevents the browser tab from sleeping.</i><br>
<audio src="https://raw.githubusercontent.com/anars/blank-audio/master/10-minutes-of-silence.mp3" autoplay loop controls style="width: 300px;" />

In [ ]:
#@title 4. Start ComfyUI Session (via Ngrok - Fiable)
import subprocess
import os
import sys

# 1. --- METTEZ VOTRE TOKEN NGROK ICI ---
NGROK_TOKEN = "3FfYHZzmoLXqqjNRxGqVWYjEjXx_59R6wvMnrwCagaBmS5gxK"
# ---------------------------------------

WORKSPACE = "/content/ComfyUI"

if os.path.exists(os.path.join(WORKSPACE, "main.py")):
    os.chdir(WORKSPACE)

    if NGROK_TOKEN == "VOTRE_TOKEN_ICI":
        print("❌ ERREUR : Vous devez d'abord coller votre token Ngrok à la ligne 7 du code !")
    else:
        # Installation de pyngrok
        print("[SYSTEM] Installation de Ngrok...")
        subprocess.run([sys.executable, "-m", "pip", "install", "pyngrok"], capture_output=True)

        from pyngrok import ngrok
        # Authentification
        ngrok.set_auth_token(NGROK_TOKEN)

        # Lancement de ComfyUI
        print("[SYSTEM] Booting ComfyUI...")
        cmd = ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"]
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

        # Création du tunnel
        print("\n[SYSTEM] Démarrage du tunnel Ngrok...")
        public_url = ngrok.connect(8188).public_url

        print("\n" + "="*70)
        print(f"🚀 COMFYUI EST DISPONIBLE ICI : {public_url}")
        print("="*70 + "\n")

        # Affichage des logs en direct
        for line in iter(process.stdout.readline, ''):
            sys.stdout.write(line)
            sys.stdout.flush()
else:
    print("[FATAL ERROR] Dossier ComfyUI introuvable. Vérifiez l'installation.")

[SYSTEM] Installation de Ngrok...
[SYSTEM] Booting ComfyUI...

[SYSTEM] Démarrage du tunnel Ngrok...

🚀 COMFYUI EST DISPONIBLE ICI : https://giggle-copy-litigator.ngrok-free.dev

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.12.13 environment at: /usr
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-06-30 13:05:09.132
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager

[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_int8_convrot_weight', 'dequantize_int8_convrot_weight_dtype', 'dequantize_int8_simple', 'dequantize_int8_simple_dtype', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'gemv_awq_w4a16', 'int8_linear', 'quantize_and_rotate_rowwise', 'quantize_int8_convrot_weight', 'quantize_int8_rowwise', 'quantize_int8_tensorwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'quantize_svdquant_w4a4', 'scaled_mm_nvfp4', 'scaled_mm_svdquant_w4a4', 'stochastic_rounding_fp8']}
[INFO] Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_sp

[INFO] xformers version: 0.0.35
[INFO] Set vram state to: NORMAL_VRAM
[INFO] Device: cuda:0 Tesla T4 : cudaMallocAsync
[INFO] Using async weight offloading with 2 streams
[INFO] Enabled pinned memory 11677.0
[INFO] Using xformers attention


aimdo: /project/src-posix/cuda-funchooks.c:52:DEBUG:aimdo_setup_hooks: hooks successfully installed
aimdo: /project/src/control.c:247:INFO:comfy-aimdo inited for GPU: Tesla T4 (VRAM: 14912 MB)
[INFO] DynamicVRAM support detected and enabled
[INFO] Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[INFO] ComfyUI version: 0.26.0
[INFO] comfy-aimdo version: 0.4.10
[INFO] comfy-kitchen version: 0.2.15
[INFO] comfyui-frontend-package version: 1.45.20
[INFO] comfyui-workflow-templates version: 0.10.7
[INFO] comfyui-embedded-docs version: 0.5.6
[INFO] comfy-kitchen version: 0.2.15
[INFO] comfy-aimdo version: 0.4.10
[INFO] [Prompt Server] web root: /usr/local/lib/python3.12/dist-packages/comfyui_frontend_package/static
[INFO] Asset seeder disabled
[INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


13:05:21.371 I [promptchain] registered native preprocessors 'pcr_cnaux' (7 nodes)
13:05:21.371 I [promptchain] registered native preprocessors 'pcr_rtmpose' (1 nodes)
[INFO] ### Loading: ComfyUI-Manager (V3.41)
[INFO] [ComfyUI-Manager] network_mode: public
[INFO] [ComfyUI-Manager] ComfyUI per-queue preview override detected (PR #11261). Manager's preview method feature is disabled. Use ComfyUI's --preview-method CLI option or 'Settings > Execution > Live preview method'.
[PromptChain] Loaded 3 model(s) from index
[INFO] ### ComfyUI Version: v0.26.0-22-gba3f697d | Released on '2026-06-30'


[PromptChain]   Danbooru: 140,724 tags
[PromptChain]   Derpibooru: 96,844 tags
[PromptChain]   E621: 101,564 tags
[PromptChain]   QwenEdit: 120 tags
[PromptChain]   SDXL: 159 tags
[PromptChain] Loaded 5 tag source(s) from /content/ComfyUI/user/PromptChain/tags
[INFO] 
Import times for custom nodes:
[INFO]    0.1 seconds: /content/ComfyUI/custom_nodes/comfyui-promptchain
[INFO]    0.4 seconds: /content/ComfyUI/custom_nodes/comfyui_ipadapter_plus
[INFO]    0.5 seconds: /content/ComfyUI/custom_nodes/websocket_image_save.py
[INFO]    1.7 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager
[INFO] 
[INFO] Context impl SQLiteImpl.
[INFO] Will assume non-transactional DDL.
[INFO] [ComfyUI-Manager] default cache updated: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/model-list.json
[INFO] [ComfyUI-Manager] default cache updated: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/github-stats.json
[INFO] [ComfyUI-Manager] default cache updated: https://raw.githu

In [ ]:
#@title 5. Build Tarball Cache & Backup
#@markdown Compresses your local UI nodes into a single, high-speed archive and saves it to Drive. **Run this before disconnecting** so your next boot is instant.

import os
print("🔄 Compressing UI workspace into high-speed cache...")
WORKSPACE = "/content/ComfyUI"
CACHE_TAR = "/content/drive/MyDrive/ComfyUI/comfy_ui_cache.tar"

if os.path.exists(WORKSPACE):
    os.chdir(WORKSPACE)
    !tar -cf "{CACHE_TAR}" custom_nodes user
    print(f"✅ Cache built and saved to: {CACHE_TAR}")
    print("✅ You may now safely disconnect your session.")
else:
    print("⚠️ ComfyUI workspace not found. Nothing to backup.")

In [ ]:
!nvidia-smi